### Task 2: Retrieval-Augmented Generation (RAG) from Scratch
Building a document-based question-answering system using embeddings, NumPy, BM25, and FAISS.

### 1. Problem Statement
The objective is to build a question-answering system that retrieves relevant information from a public document collection and generates context-based answers.

### 2. Document Loading and Preprocessing
Loading the 48-page NIST AI Risk Management Framework PDF and cleaning extracted text for retrieval.

In [1]:
import os, re, math, json, hashlib, bisect, unicodedata, warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import numpy as np, pandas as pd
from scipy import sparse

WORKDIR = Path(os.environ.get("RAG_WORKDIR", "./rag_workdir"))
(WORKDIR / "pdf").mkdir(parents=True, exist_ok=True)
(WORKDIR / "cache").mkdir(parents=True, exist_ok=True)


DOCS = [
    {"doc_id": "nist_ai_rmf_1_0", "url": "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"},
]

CHUNK_WORDS   = (120, 250)    
OVERLAP_WORDS = 30
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"                       
QUERY_PREFIX  = "Represent this sentence for searching relevant passages: " 
RERANK_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"        
N_CANDIDATES  = 50            
RANK_DEPTH    = 20            
COVER_THRESH  = 0.8           
BUDGET_WORDS  = 1000          

In [2]:
_TRANS = {0x2018: "'", 0x2019: "'", 0x201c: '"', 0x201d: '"', 0x2013: "-", 0x2014: "-", 0x00ad: None}
def norm_text(s: str) -> str:
    return unicodedata.normalize("NFKC", s).translate(_TRANS)

def clean_pages(raw_pages):
    lines = [[l.strip() for l in p.splitlines() if l.strip()] for p in raw_pages]
    edge = lambda ls: set(ls[:3] + ls[-3:])                            
    freq = Counter(l for ls in lines for l in edge(ls) if len(l) < 100)
    thresh = max(3, int(0.3 * len(raw_pages)))
    boiler = {l for l, c in freq.items() if c >= thresh}            
    page_no = re.compile(r"^(page\s*)?\d{1,3}$", re.I)
    kept = []
    for ls in lines:
        e = len(ls)
        kept.append("\n".join(l for i, l in enumerate(ls)
                              if not ((i < 3 or i >= e - 3) and (l in boiler or page_no.match(l)))))
    kept = [norm_text(p) for p in kept]
    vocab = Counter(re.findall(r"[a-z]+", " ".join(kept).lower()))
    def join_hyphen(m):                                                # "perfor-\nmance" -> "performance", "third-\nparty" -> "third-party"
        joined = m.group(1) + m.group(2)
        return joined if vocab[joined.lower()] > 0 else m.group(1) + "-" + m.group(2)
    kept = [re.sub(r"(\w+)-\n(\w+)", join_hyphen, p) for p in kept]
    return [re.sub(r"\s+", " ", p).strip() for p in kept]

@dataclass
class Doc:
    doc_id: str
    text: str
    page_starts: list
    page_nos: list
    n_pages: int
    def page_of(self, pos: int) -> int:
        return self.page_nos[max(bisect.bisect_right(self.page_starts, pos) - 1, 0)]

def build_doc(doc_id, raw_pages) -> Doc:
    pages = clean_pages(raw_pages)
    parts, starts, nos, pos = [], [], [], 0
    for i, t in enumerate(pages, 1):
        if len(t) < 40:                                                # blank / figure-only pages
            continue
        starts.append(pos); nos.append(i); parts.append(t); pos += len(t) + 1
    return Doc(doc_id, " ".join(parts), starts, nos, len(raw_pages))

In [3]:
!pip install PyMuPDF requests

   ---------------------------------------- 0.0/63.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/63.0 MB ? eta -:--:--
    --------------------------------------- 1.0/63.0 MB 4.2 MB/s eta 0:00:15
   - -------------------------------------- 2.6/63.0 MB 5.6 MB/s eta 0:00:11
   -- ------------------------------------- 3.7/63.0 MB 5.6 MB/s eta 0:00:11
   --- ------------------------------------ 5.2/63.0 MB 6.1 MB/s eta 0:00:10
   ---- ----------------------------------- 6.8/63.0 MB 6.4 MB/s eta 0:00:09
   ----- ---------------------------------- 8.7/63.0 MB 6.7 MB/s eta 0:00:09
   ------ --------------------------------- 10.5/63.0 MB 7.0 MB/s eta 0:00:08
   ------- -------------------------------- 12.3/63.0 MB 7.1 MB/s eta 0:00:08
   -------- ------------------------------- 13.9/63.0 MB 7.3 MB/s eta 0:00:07
   --------- ------------------------------ 15.5/63.0 MB 7.3 MB/s eta 0:00:07
   ----------- ---------------------------- 17.8/63.0 MB 7.6 MB/s eta 0:00:06
   -----


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import requests, fitz

def load_pdf(spec) -> Doc:
    path = WORKDIR / "pdf" / f"{spec['doc_id']}.pdf"
    if not path.exists():
        r = requests.get(spec["url"], timeout=120, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        path.write_bytes(r.content)
    with fitz.open(path) as pdf:
        return build_doc(spec["doc_id"], [p.get_text("text") for p in pdf])

docs = {s["doc_id"]: load_pdf(s) for s in DOCS}
total_pages = sum(d.n_pages for d in docs.values())
for d in docs.values():
    print(f"{d.doc_id}: {d.n_pages} pages, {len(d.text):,} chars")
assert total_pages >= 30, "Task requires at least 30 pages"

c:\Users\Shraddha\AppData\Local\Programs\Python\Python311\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


nist_ai_rmf_1_0: 48 pages, 104,604 chars


In [5]:
_SENT_END = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(\[])")

def sentence_spans(text):
    spans, start = [], 0
    for m in _SENT_END.finditer(text):
        spans.append((start, m.start())); start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans

def pack_spans(text, target_words, overlap_words):
    units = []                                                         # (start, end, n_words)
    for s, e in sentence_spans(text):
        ws = [m.span() for m in re.finditer(r"\S+", text[s:e])]
        if len(ws) <= target_words:
            units.append((s, e, len(ws)))
        else:
            for a in range(0, len(ws), target_words):
                b = min(a + target_words, len(ws)) - 1
                units.append((s + ws[a][0], s + ws[b][1], b - a + 1))
    out, i, n = [], 0, len(units)
    while i < n:
        j, w = i, 0
        while j < n and (j == i or w + units[j][2] <= target_words):
            w += units[j][2]; j += 1
        out.append((units[i][0], units[j - 1][1]))
        if j >= n:
            break
        k, ow = j, 0
        while k > i + 1 and ow < overlap_words:                         # step back to create overlap
            k -= 1; ow += units[k][2]
        i = k
    return out

def chunk_corpus(docs, target_words, overlap_words) -> pd.DataFrame:
    rows = []
    for d in docs.values():
        for s, e in pack_spans(d.text, target_words, overlap_words):
            t = d.text[s:e]
            rows.append((d.doc_id, s, e, d.page_of(s), d.page_of(e - 1), len(t.split()), t))
    df = pd.DataFrame(rows, columns=["doc_id", "start", "end", "page_start", "page_end", "n_words", "text"])
    df.insert(0, "chunk_id", np.arange(len(df)))
    return df

In [6]:
class Embedder:
    def __init__(self, name=EMBED_MODEL, query_prefix=QUERY_PREFIX, batch_size=64):
        from sentence_transformers import SentenceTransformer
        self.name, self.query_prefix, self.batch_size = name, query_prefix, batch_size
        self.model = SentenceTransformer(name)
        self.max_len = self.model.max_seq_length
    def encode_docs(self, texts):
        return self.model.encode(texts, batch_size=self.batch_size, normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=len(texts) > 500).astype("float32")
    def encode_queries(self, queries):
        return self.model.encode([self.query_prefix + q for q in queries], normalize_embeddings=True,
                                 convert_to_numpy=True).astype("float32")
    def truncation_report(self, texts, sample=2000):
        rng = np.random.default_rng(0)
        pick = [texts[i] for i in rng.choice(len(texts), min(sample, len(texts)), replace=False)]
        lens = np.array([len(x) for x in self.model.tokenizer(pick, truncation=False)["input_ids"]])
        return {"max_len": self.max_len, "mean_tokens": float(lens.mean()), "p95_tokens": float(np.percentile(lens, 95)),
                "pct_truncated": float((lens > self.max_len).mean() * 100)}

def embed_with_cache(texts, embedder, tag, shard=20000):
    parts = []
    for s in range(0, len(texts), shard):
        part = texts[s:s + shard]
        h = hashlib.sha1(embedder.name.encode())
        for t in part: h.update(t.encode())
        path = WORKDIR / "cache" / f"{tag}_{s}_{h.hexdigest()[:16]}.npy"
        if path.exists():
            parts.append(np.load(path))
        else:
            v = embedder.encode_docs(part); np.save(path, v); parts.append(v)
    return np.vstack(parts)

In [7]:
def topk(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]                          # O(n), not O(n log n)
    return idx[np.argsort(-scores[idx])]

class VectorIndex:
    def __init__(self, vecs, use_faiss=True, hnsw_threshold=200_000):
        self.vecs = np.ascontiguousarray(vecs, dtype="float32")
        self.n, self.d = self.vecs.shape
        self.kind, self.faiss = "numpy", None
        if use_faiss:
            try:
                import faiss
                if self.n < hnsw_threshold:
                    ix, self.kind = faiss.IndexFlatIP(self.d), "faiss-flat"
                else:
                    ix = faiss.IndexHNSWFlat(self.d, 32, faiss.METRIC_INNER_PRODUCT)
                    ix.hnsw.efConstruction, ix.hnsw.efSearch, self.kind = 200, 128, "faiss-hnsw"
                ix.add(self.vecs); self.faiss = ix
            except ImportError:
                pass
    def search(self, q, k):
        q = np.ascontiguousarray(np.atleast_2d(q), dtype="float32"); k = min(k, self.n)
        if self.faiss is not None:
            return self.faiss.search(q, k)
        S = q @ self.vecs.T
        idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
        sc = np.take_along_axis(S, idx, 1)
        o = np.argsort(-sc, axis=1)
        return np.take_along_axis(sc, o, 1), np.take_along_axis(idx, o, 1)

_STOP = set("a an and are as at be been by for from has have in is it its of on or that the their this to was were which with within into than then these those such can may not also".split())
def bm25_tokens(text):
    out = []
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if w in _STOP: continue
        if len(w) > 4 and w.endswith("ies"): w = w[:-3] + "y"
        elif len(w) > 3 and w.endswith("s") and not w.endswith("ss"): w = w[:-1]
        out.append(w)
    return out

class BM25:
    def __init__(self, k1=1.5, b=0.75): self.k1, self.b = k1, b
    def fit(self, texts):
        vocab, rows, cols, vals = {}, [], [], []
        for r, t in enumerate(texts):
            for term, c in Counter(bm25_tokens(t)).items():
                rows.append(r); cols.append(vocab.setdefault(term, len(vocab))); vals.append(c)
        n, V = len(texts), len(vocab)
        tf = sparse.csr_matrix((vals, (rows, cols)), shape=(n, V), dtype="float32")
        dl = np.asarray(tf.sum(axis=1)).ravel(); avg = dl.mean()
        df = np.bincount(tf.indices, minlength=V)
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype("float32")
        coo = tf.tocoo()
        w = coo.data * (self.k1 + 1) / (coo.data + self.k1 * (1 - self.b + self.b * dl[coo.row] / avg))
        self.W = sparse.csc_matrix((w * idf[coo.col], (coo.row, coo.col)), shape=(n, V))
        self.vocab, self.n = vocab, n
        return self
    def search(self, query, k):
        ids = [self.vocab[t] for t in set(bm25_tokens(query)) if t in self.vocab]
        if not ids: return np.array([], dtype=int), np.array([], dtype="float32")
        scores = np.asarray(self.W[:, ids].sum(axis=1)).ravel()
        top = topk(scores, k); top = top[scores[top] > 0]
        return top, scores[top]

def rrf(rank_lists, k=60):
    fused = {}
    for ranking in rank_lists:
        for r, cid in enumerate(ranking):
            fused[int(cid)] = fused.get(int(cid), 0.0) + 1.0 / (k + r + 1)
    return sorted(fused.items(), key=lambda x: -x[1])

class Reranker:
    def __init__(self, name=RERANK_MODEL):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder(name, max_length=512)
    def score(self, query, texts):
        return self.model.predict([(query, t) for t in texts], batch_size=32, show_progress_bar=False)

class Retriever:
    def __init__(self, chunks, embedder, vecs, reranker=None):
        self.chunks = chunks.reset_index(drop=True)
        self.embedder, self.reranker = embedder, reranker
        self.vindex = VectorIndex(vecs)
        self.bm25 = BM25().fit(self.chunks["text"].tolist())
    def search(self, query, k=5, mode="hybrid", rerank=False, n_candidates=N_CANDIDATES):
        n = max(n_candidates, k)
        if mode in ("dense", "hybrid"):
            ds, di = self.vindex.search(self.embedder.encode_queries([query]), n)
            dense = [(int(i), float(s)) for i, s in zip(di[0], ds[0]) if i >= 0]
        if mode in ("bm25", "hybrid"):
            bi, bs = self.bm25.search(query, n)
            sparse_hits = [(int(i), float(s)) for i, s in zip(bi, bs)]
        if   mode == "dense": cand = dense
        elif mode == "bm25":  cand = sparse_hits
        else:                 cand = rrf([[i for i, _ in dense], [i for i, _ in sparse_hits]])
        cand = cand[:n]
        if rerank and cand:
            sc = self.reranker.score(query, [self.chunks.at[i, "text"] for i, _ in cand])
            cand = sorted([(i, float(s)) for (i, _), s in zip(cand, sc)], key=lambda x: -x[1])
        return [{**self.chunks.loc[i, ["chunk_id", "doc_id", "start", "end", "page_start", "page_end", "n_words", "text"]].to_dict(),
                 "score": s} for i, s in cand[:k]]

In [8]:
import re
import pandas as pd

GOLD_DRAFT = [
    ("Which top-level activities make up the core of the framework?",
     "four functions: GOVERN, MAP, MEASURE, and MANAGE"),

    ("Which activity runs through everything instead of being a separate stage?",
     "cross-cutting function"),

    ("What qualities make an AI system trustworthy?",
     "valid and reliable, safe, secure and resilient"),

    ("Why is it hard to quantify risk when an organisation relies on outside vendors' AI?",
     "third-party software, hardware, and data"),

    ("How does the document define risk?",
     "composite measure of an event"),

    ("How does the document define an AI system?",
     "engineered or machine-based system"),

    ("How much risk is an organisation willing to accept?",
     "readiness to bear the risk"),

    ("What does TEVV stand for and why does it matter?",
     "test, evaluation, verification, and validation"),

    ("What goes wrong if training data does not reflect the setting where the system is used?",
     "true or appropriate representation"),

    ("What are profiles for?",
     "use-case profiles"),

    ("What is the difference between explainability and interpretability?",
     "mechanisms underlying AI systems"),

    ("Is it realistic to aim for zero risk?",
     "eliminate negative risk entirely"),

    ("When will the framework itself be revisited?",
     "NIST will review the content and usefulness of the Framework regularly"),

    ("What does the function about acting on risks do with resources?",
     "allocating risk resources"),

    ("How should the human role in AI decisions be handled?",
     "human oversight"),
]



def find(keywords, max_hits=8, width=220):

    """Find sentences containing all specified keywords."""

    kws = [k.lower() for k in keywords.split()]
    n = 0

    for d in docs.values():

        for s, e in sentence_spans(d.text):

            sentence = d.text[s:e].lower()

            if all(k in sentence for k in kws):

                print(
                    f"[{d.doc_id} p.{d.page_of(s)}] "
                    f"{d.text[s:e][:width]}\n"
                )

                n += 1

                if n >= max_hits:
                    return



def locate_gold(items, docs):

    gold = []
    problems = []

    sent_cache = {
        d.doc_id: sentence_spans(d.text)
        for d in docs.values()
    }

    for n, (question, anchor) in enumerate(items, 1):

        anchor = norm_text(anchor)

        pattern = re.compile(
            r"\s+".join(map(re.escape, anchor.split())),
            re.I
        )

        matches = [
            (d, m)
            for d in docs.values()
            for m in pattern.finditer(d.text)
        ]

        if not matches:

            anchor_words = set(bm25_tokens(anchor))
            candidates = []

            for d in docs.values():

                for s, e in sent_cache[d.doc_id]:

                    sentence = d.text[s:e]

                    overlap = len(
                        anchor_words &
                        set(bm25_tokens(sentence))
                    )

                    if overlap:

                        candidates.append((
                            overlap,
                            d,
                            s,
                            e
                        ))

            candidates.sort(
                key=lambda x: x[0],
                reverse=True
            )

            problems.append(
                (n, question, anchor, candidates[:3])
            )

            continue

        d, match = matches[0]

        if len(matches) > 1:

            print(
                f"Q{n}: {len(matches)} matches found. "
                f"Using first occurrence."
            )

        s0, e0 = next(
            (
                (s, e)
                for s, e in sent_cache[d.doc_id]
                if s <= match.start() < e
            ),
            (match.start(), match.end())
        )

        e0 = max(e0, match.end())

        gold.append({

            "qid": n,

            "question": question,

            "doc_id": d.doc_id,

            "start": s0,

            "end": e0,

            "page": d.page_of(s0),

            "passage": d.text[s0:e0]

        })


    if problems:

        for n, question, anchor, candidates in problems:

            print(
                f"\nQ{n}: Anchor not found: {anchor}"
            )

            print("Question:", question)

            for overlap, d, s, e in candidates:

                print(
                    f"Candidate (p.{d.page_of(s)}): "
                    f"{d.text[s:e][:250]}"
                )

        raise ValueError(
            f"{len(problems)} anchors need correction."
        )

    return gold



gold = locate_gold(GOLD_DRAFT, docs)

assert len(gold) == 15, "Expected exactly 15 gold questions."

print("\nSuccessfully extracted all 15 gold passages!\n")



for g in gold:

    print(
        f"Q{g['qid']}. {g['question']}\n"
        f"   Document: {g['doc_id']}\n"
        f"   PDF page: {g['page']}\n"
        f"   Gold passage: {g['passage'][:300]}\n"
    )


gold_df = pd.DataFrame(gold)

gold_df[
    ["qid", "question", "page", "passage"]
].to_csv(
    WORKDIR / "gold_questions.csv",
    index=False
)

print("Gold evaluation dataset saved successfully!")

print("Total questions:", len(gold_df))

print("Output file:", WORKDIR / "gold_questions.csv")

Q2: 3 matches found. Using first occurrence.
Q3: 2 matches found. Using first occurrence.
Q8: 3 matches found. Using first occurrence.
Q15: 3 matches found. Using first occurrence.

Successfully extracted all 15 gold passages!

Q1. Which top-level activities make up the core of the framework?
   Document: nist_ai_rmf_1_0
   PDF page: 25
   Gold passage: As illus-trated in Figure 5, the Core is composed of four functions: GOVERN, MAP, MEASURE, and MANAGE.

Q2. Which activity runs through everything instead of being a separate stage?
   Document: nist_ai_rmf_1_0
   PDF page: 5
   Gold passage: Governance is designed to be a cross-cutting function to inform and be infused throughout the other three functions.

Q3. What qualities make an AI system trustworthy?
   Document: nist_ai_rmf_1_0
   PDF page: 7
   Gold passage: Next, AI risks and trustworthiness are analyzed, outlining the characteristics of trustworthy AI systems, which include valid and reliable, safe, secure and resilient, acco

In [9]:
embedder = Embedder()
reranker = Reranker()
RETRIEVERS = {}
for size in CHUNK_WORDS:
    chunks = chunk_corpus(docs, size, OVERLAP_WORDS)
    vecs = embed_with_cache(chunks["text"].tolist(), embedder, f"w{size}")
    RETRIEVERS[size] = Retriever(chunks, embedder, vecs, reranker)
    rep = embedder.truncation_report(chunks["text"].tolist())
    print(f"{size}-word chunks: {len(chunks)} chunks | avg {chunks.n_words.mean():.0f} words | index={RETRIEVERS[size].vindex.kind}")
    print(f"   token audit: mean {rep['mean_tokens']:.0f}, p95 {rep['p95_tokens']:.0f}, window {rep['max_len']}, truncated {rep['pct_truncated']:.1f}%")

c:\Users\Shraddha\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Shraddha\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Shraddha\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (12 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

120-word chunks: 241 chunks | avg 103 words | index=faiss-flat
   token audit: mean 134, p95 163, window 512, truncated 0.0%
250-word chunks: 81 chunks | avg 232 words | index=faiss-flat
   token audit: mean 298, p95 334, window 512, truncated 0.0%


In [10]:
def is_hit(hit, g, thr=COVER_THRESH):
    if hit["doc_id"] != g["doc_id"]: return False
    overlap = min(hit["end"], g["end"]) - max(hit["start"], g["start"])
    return overlap >= thr * (g["end"] - g["start"])

def evaluate(retr, gold, mode, rerank, depth=RANK_DEPTH):
    rows = []
    for g in gold:
        hits = retr.search(g["question"], k=depth, mode=mode, rerank=rerank)
        rank = next((r for r, h in enumerate(hits, 1) if is_hit(h, g)), np.nan)
        rows.append({"qid": g["qid"], "question": g["question"], "gold_page": g["page"], "rank": rank,
                     "top_pages": [h["page_start"] for h in hits[:5]], "hits": hits})
    return pd.DataFrame(rows)

def summarize(df, avg_words, budget=BUDGET_WORDS):
    r = df["rank"]
    rec = lambda k: float((r <= k).mean())
    k_bud = max(1, round(budget / avg_words))
    return {"R@1": rec(1), "R@3": rec(3), "R@5": rec(5), "R@10": rec(10),
            "MRR@10": float(np.where(r <= 10, 1 / r, 0).mean()),
            "avg_words_in_top5": 5 * avg_words, f"R@~{budget}w (k={k_bud})": rec(k_bud)}

METHODS = {"dense": ("dense", False), "bm25": ("bm25", False),
           "hybrid (dense+BM25, RRF)": ("hybrid", False), "hybrid + re-rank": ("hybrid", True)}
RESULTS, rows = {}, []
for size, retr in RETRIEVERS.items():
    avg = retr.chunks["n_words"].mean()
    for name, (mode, rr) in METHODS.items():
        RESULTS[(size, name)] = evaluate(retr, gold, mode, rr)
        rows.append({"chunk_words": size, "method": name, **summarize(RESULTS[(size, name)], avg)})
summary = pd.DataFrame(rows)
summary.to_csv(WORKDIR / "retrieval_summary.csv", index=False)
summary.round(3)

,chunk_words,method,R@1,R@3,R@5,R@10,MRR@10,avg_words_in_top5,R@~1000w (k=10),R@~1000w (k=4)
0,120,dense,0.333,0.667,0.667,0.800,0.507,515.560,0.800,NaN
1,120,bm25,0.200,0.267,0.333,0.400,0.261,515.560,0.400,NaN
2,120,"hybrid (dense+BM25, RRF)",0.333,0.400,0.400,0.533,0.374,515.560,0.533,NaN
3,120,hybrid + re-rank,0.467,0.667,0.800,0.867,0.595,515.560,0.867,NaN
4,250,dense,0.400,0.667,0.667,0.800,0.540,1158.086,NaN,0.667
5,250,bm25,0.133,0.333,0.333,0.533,0.252,1158.086,NaN,0.333
6,250,"hybrid (dense+BM25, RRF)",0.267,0.467,0.533,0.800,0.375,1158.086,NaN,0.467
7,250,hybrid + re-rank,0.533,0.600,0.800,0.933,0.628,1158.086,NaN,0.667


In [11]:
def show_failures(size, method, max_cases=3, depth_shown=3):
    df = RESULTS[(size, method)]
    bad = df[~(df["rank"] <= 5)]
    print(f"=== {size}-word chunks | {method} | {len(bad)} of {len(df)} questions miss top-5 ===\n")
    for _, r in bad.head(max_cases).iterrows():
        g = next(x for x in gold if x["qid"] == r["qid"])
        print(f"Q{r['qid']}: {r['question']}\n  GOLD (p.{g['page']}): {g['passage'][:350]}")
        print(f"  first correct rank: {r['rank']}")
        for h in r["hits"][:depth_shown]:
            print(f"  got  (p.{h['page_start']}, score {h['score']:.3f}): {h['text'][:200]}")
        print()

for size in CHUNK_WORDS:
    show_failures(size, "dense")

=== 120-word chunks | dense | 5 of 15 questions miss top-5 ===

Q2: Which activity runs through everything instead of being a separate stage?
  GOLD (p.5): Governance is designed to be a cross-cutting function to inform and be infused throughout the other three functions.
  first correct rank: nan
  got  (p.25, score 0.600): Each of these high-level functions is broken down into categories and subcategories. Categories and subcategories are subdivided into specific actions and outcomes. Actions do not constitute a checkli
  got  (p.26, score 0.589): Assuming a governance structure is in place, functions may be performed in any order across the AI lifecycle as deemed to add value by a user of the framework. After instituting the outcomes in GOVERN
  got  (p.26, score 0.587): Along with the AI RMF, the Playbook is part of the NIST Trustworthy and Responsible AI Resource Center. Framework users may apply these functions as best suits their needs for managing AI risks based 

Q3: What qual

In [12]:
def build_prompt(question, hits):
    ctx = "\n\n".join(f"[{h['doc_id']} | p.{h['page_start']}] {h['text']}" for h in hits)
    return ("Answer the question using ONLY the context. Cite sources as [doc | page]. "
            "If the context does not contain the answer, reply exactly: I could not find this in the provided documents.\n\n"
            f"CONTEXT:\n{ctx}\n\nQUESTION: {question}\nANSWER:")

def answer(question, retr, llm=None, k=5, mode="hybrid", rerank=True):
    hits = retr.search(question, k=k, mode=mode, rerank=rerank)
    if llm is None:                                                   # extractive fallback: no API key needed
        sents = [(s, h) for h in hits for s in re.split(r"(?<=[.!?])\s+", h["text"])]
        qv = retr.embedder.encode_queries([question])[0]
        sv = retr.embedder.encode_docs([s for s, _ in sents])
        best = int(np.argmax(sv @ qv))
        return {"answer": sents[best][0], "sources": [(h["doc_id"], h["page_start"]) for h in hits]}
    return {"answer": llm(build_prompt(question, hits)), "sources": [(h["doc_id"], h["page_start"]) for h in hits]}

print(answer("Which top-level activities make up the core of the framework?", RETRIEVERS[CHUNK_WORDS[0]]))

{'answer': 'Part 2 comprises the "Core" of the Framework.', 'sources': [('nist_ai_rmf_1_0', 24), ('nist_ai_rmf_1_0', 25), ('nist_ai_rmf_1_0', 7), ('nist_ai_rmf_1_0', 14), ('nist_ai_rmf_1_0', 7)]}
